# ASSIGNMENT 2 - DATA PROCESSING

# Phase 0 - Project set up

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler


# Phase 1 – Data Loading and Initial Inspection


## 1.1 Load dataset


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("bs140513_032310.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'bs140513_032310.csv'

## 1.2 Inspect structure


In [ ]:
df.head()
df.info()
df.describe()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 594643 entries, 0 to 594642
Data columns (total 10 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   step         594643 non-null  int64  
 1   customer     594643 non-null  object 
 2   age          594643 non-null  object 
 3   gender       594643 non-null  object 
 4   zipcodeOri   594643 non-null  object 
 5   merchant     594643 non-null  object 
 6   zipMerchant  594643 non-null  object 
 7   category     594643 non-null  object 
 8   amount       594643 non-null  float64
 9   fraud        594643 non-null  int64  
dtypes: float64(1), int64(2), object(7)
memory usage: 45.4+ MB


,step,amount,fraud
count,594643.000000,594643.000000,594643.000000
mean,94.986827,37.890135,0.012108
std,51.053632,111.402831,0.109369
min,0.000000,0.000000,0.000000
25%,52.000000,13.740000,0.000000
50%,97.000000,26.900000,0.000000
75%,139.000000,42.540000,0.000000
max,179.000000,8329.960000,1.000000


# Phase2 - Basic Cleaning

## 2.1 Remove duplicates

In [ ]:
df = df.drop_duplicates()

## 2.2 Handle missing values

In [ ]:
df['amount'] = df['amount'].fillna(df['amount'].median())

df['merchant'] = df['merchant'].fillna(df['merchant'].mode()[0])
df['category'] = df['category'].fillna(df['category'].mode()[0])

In [ ]:
df.columns = df.columns.str.strip()
print(df.columns)

Index(['step', 'customer', 'age', 'gender', 'zipcodeOri', 'merchant',
       'zipMerchant', 'category', 'amount', 'fraud'],
      dtype='object')


In [ ]:
df.isnull().sum()

step           0
customer       0
age            0
gender         0
zipcodeOri     0
merchant       0
zipMerchant    0
category       0
amount         0
fraud          0
dtype: int64

## 2.3 Remove invalid values

In [ ]:
df['fraud'] = df['fraud'].astype(int)

# PHASE 3 - Data Type Correction & Sorting (CRITICAL)

In [ ]:
df = df.sort_values(['customer', 'step']).copy()

# PHASE 4 - Sequential Historical Features (Past-Only)

## 4.1 Rolling Historical Mean (Past Only)

In [ ]:
df['rolling_mean_10'] = (
    df.groupby('customer')['amount']
      .transform(lambda x: x.shift(1).rolling(10, min_periods=1).mean())
)

## 4.2 Rolling Historical Standard Deviation (Past Only)

In [ ]:
df['rolling_std_10'] = (
    df.groupby('customer')['amount']
      .transform(lambda x: x.shift(1).rolling(10, min_periods=1).std())
)


## 4.3 Amount Z-Score Deviation

In [ ]:
df['amount_zscore'] = (
    (df['amount'] - df['rolling_mean_10']) /
    (df['rolling_std_10'] + 1e-6)
)

# PHASE 5 - Novelty & Behavioral Change Features

## 5.1 New Merchant Detection

In [ ]:
df['new_merchant'] = (
    df.groupby('customer')['merchant']
      .transform(lambda x: ~x.duplicated())
      .astype(int)
)

## 5.2 Category Novelty

In [ ]:
df['new_category'] = (
    df.groupby('customer')['category']
      .transform(lambda x: ~x.duplicated())
      .astype(int)
)

# PHASE 6 - Rolling Transaction Burst Detection

In [ ]:
df['merchant_repeat'] = (df['merchant_freq'] > 0).astype(int)

In [ ]:
df['merchant_freq'] = (
    df.groupby(['customer', 'merchant']).cumcount()
)

# PHASE 7 - Anomaly Flag & Counters

## 7.1 Category frequency per customer

In [ ]:
df['category_freq'] = (
    df.groupby(['customer', 'category']).cumcount()
)

## 7.2 Category deviation flag

In [ ]:
df['category_rare'] = (df['category_freq'] < 2).astype(int)

# PHASE 8 - Repetition Tracking

In [ ]:
df['tx_count_10'] = (
    df.groupby('customer')['amount']
      .transform(lambda x: x.shift(1).rolling(10, min_periods=1).count())
)

In [ ]:
df['burst_flag'] = (df['tx_count_10'] > 8).astype(int)

# PHASE 9 - Log Transformation (Reduce Skewness)

## 9.1 High Z-Score Flag

In [ ]:
df['high_zscore_flag'] = (df['amount_zscore'].abs() > 3).astype(int)

## 9.2 Rolling anomaly counter

In [ ]:
df['anomaly_count_20'] = (
    df.groupby('customer')['high_zscore_flag']
      .transform(lambda x: x.shift(1).rolling(20, min_periods=1).sum())
)

# PHASE 10 - Categorical Encoding

In [ ]:
df = pd.get_dummies(df, columns=['gender', 'category'], drop_first=True)

# PHASE 11 - Feature Scaling

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('fraud', axis=1)
y = df['fraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=False
)

# PHASE 12 - Final Cleaning Before Modeling

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

num_cols = X_train.select_dtypes(include=np.number).columns

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])